# Blender Dataset — Evaluation Results

Compares four matchers on the Blender multi-object synthetic dataset:

| Model | Abbreviation |
|---|---|
| SuperPoint + LightGlue | SP+LG |
| SuperPoint + SuperGlue | SP+SG |
| SuperPoint + LSD + GlueStick | SP+GS |
| LoFTR (dense) | LoFTR |

**Object categories**:
- **Static** (room): object IDs 1–6
- **Dynamic** (moved objects): object IDs ≥ 7

**Vocabulary** - For every image pair we compute per-keypoint metrics split by object category (**overall / static / dynamic**):

| Metric | Definition |
|---|---|
| `tp` | Number of true positives. |
| `fp` | Number of false positives. |
| `precision` | TP / (TP + FP). |
| `recall` | TP / total GT-matchable keypoints. |
| `n_keypoints` | Number of detected keypoints. |
| `n_pred_matches` | Number of predicted matches. |
| `n_gt_matchable` | Number of GT-matchable keypoints. |
| `acc@{1,3,5}px` | Fraction of predicted matches with reproj. error ≤ t. |
| `mean_reproj` | Average reproj. error. |
| `std_reproj` | Standard deviation of reproj. error. |
| `fp1` | False positive: GT-matchable, same obj, reproj ≤ 3. |
| `fp2` | False positive: GT-matchable, cross obj, reproj ≤ 3. |
| `fp3` | False positive: GT-matchable, cross obj, reproj > 3 |
| `fp4` | False positive: GT-matchable, same obj, reproj > 3. |

**Data source**: `per_keypoint.h5` files containing per-keypoint GT matches, predicted matches, object IDs, and symmetric reprojection errors computed with object-aware projections.

In [ ]:
import numpy as np
import pandas as pd
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '1'
import cv2
import h5py
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import seaborn as sns
from PIL import Image
from pathlib import Path
BASE = Path.cwd()
# Expecting same structure as in Glue Factory
RESULTS_FOLDER = BASE / 'outputs/results/blender'  
DATASET_DIR = BASE / 'data/blender_dataset'

# Plot style
sns.set_theme(style = 'whitegrid', font_scale = 1.1)
MODEL_ORDER = ['SP+SG', 'LoFTR', 'SP+LG', 'SP+GS']
PALETTE = dict(zip(MODEL_ORDER, sns.color_palette('Set2', 4)))

MODEL_DIRS = {
    'SP+SG':  'superpoint+superglue_eval',
    'LoFTR':  'loftr_eval',
    'SP+LG':  'superpoint+lightglue_eval',
    'SP+GS':  'superpoint+lsd+gluestick_eval',
}

# Object-ID classification
STATIC_IDS  = set(range(1, 7))      # 1–6 (Room elements)
DYNAMIC_MIN = 7                     # ≥ 7 (moved objects)

# Reprojection error thresholds (pixels)
REPROJ_THRESHOLDS = [1, 3, 5]

---
#### Common Functions

In [ ]:
def draw_matches_on_pair(
    img0, img1, kpts0, kpts1, pred_m0, gt_m0, obj_ids0,
    category = 'static', max_lines = 200, ax = None, title = ''):
    """Draw predicted matches coloured green (correct) / red (incorrect)."""
    
    h0, w0 = img0.shape[:2]
    h1, w1 = img1.shape[:2]
    h = max(h0, h1)
    canvas = np.zeros((h, w0 + w1, 3), dtype = np.uint8)
    canvas[:h0, :w0] = img0
    canvas[:h1, w0:] = img1
    
    if ax is None:
        _, ax = plt.subplots(figsize = (16, 6))
    ax.imshow(canvas)
    ax.set_axis_off()
    ax.set_title(title, fontsize = 24, fontweight = 'bold')
    
    # Select keypoints in the chosen category
    if category == 'static':
        mask = np.isin(obj_ids0, list(STATIC_IDS))
    else:
        mask = obj_ids0 >= DYNAMIC_MIN
    
    idx_cat = np.where(mask)[0]
    pred_matched = pred_m0[idx_cat] >= 0
    gt_matchable = gt_m0[idx_cat] >= 0
    correct = (pred_m0[idx_cat] == gt_m0[idx_cat]) & gt_matchable & pred_matched
    
    # Draw only predicted matches
    matched_idx = idx_cat[pred_matched]
    correct_flags = correct[pred_matched]
    
    n_tp = int(correct_flags.sum())
    n_fp = int((~correct_flags).sum())
    
    # Sub-sample if too many
    if len(matched_idx) > max_lines:
        rng = np.random.default_rng()
        sel = rng.choice(len(matched_idx), max_lines, replace=False)
        matched_idx = matched_idx[sel]
        correct_flags = correct_flags[sel]
    
    for i, is_correct in zip(matched_idx, correct_flags):
        j = pred_m0[i]
        x0, y0 = kpts0[i]
        x1, y1 = kpts1[j]
        color = 'lime' if is_correct else 'red'
        ax.plot([x0, x1 + w0], [y0, y1], color = color, lw = 0.5, alpha = 0.6)
    
    txt = ax.text(5, h - 10, f'TP={n_tp}  FP={n_fp}',
                fontsize = 16, color = 'white')

    txt.set_path_effects([
        pe.withStroke(linewidth = 2.5, foreground = 'black')
    ])
    
    
def draw_separate_matches_on_pair(
    img0, img1, kpts0, kpts1, pred_m0, gt_m0, obj_ids0,
    max_lines = 200, ax = None, title = '', cat = None):
    """Draw predicted matches separately for static and dynamic objects."""
    
    h0, w0 = img0.shape[:2]
    h1, w1 = img1.shape[:2]
    h = max(h0, h1)
    canvas = np.zeros((h, w0 + w1, 3), dtype = np.uint8)
    canvas[:h0, :w0] = img0
    canvas[:h1, w0:] = img1
    
    if ax is None:
        _, ax = plt.subplots(figsize = (16, 6))
    ax.imshow(canvas)
    ax.set_axis_off()
    ax.set_title(f'{title} \u2014 {cat.capitalize()} Matches', fontsize = 24, fontweight = 'bold')
    
    if cat == 'static':
        mask = np.isin(obj_ids0, list(STATIC_IDS))
    else:
        mask = obj_ids0 >= DYNAMIC_MIN
            
    idx_cat = np.where(mask)[0]
    pred_matched = pred_m0[idx_cat] >= 0
    gt_matchable = gt_m0[idx_cat] >= 0
    correct = (pred_m0[idx_cat] == gt_m0[idx_cat]) & gt_matchable & pred_matched
        
    # Draw only predicted matches
    matched_idx = idx_cat[pred_matched]
    correct_flags = correct[pred_matched]
            
    n_tp = int(correct_flags.sum())
    n_fp = int((~correct_flags).sum())
        
    # Sub-sample if too many
    if len(matched_idx) > max_lines:
        rng = np.random.default_rng()
        sel = rng.choice(len(matched_idx), max_lines, replace = False)
        matched_idx = matched_idx[sel]
        correct_flags = correct_flags[sel]
        
    for i, is_correct in zip(matched_idx, correct_flags):
        j = pred_m0[i]
        x0, y0 = kpts0[i]
        x1, y1 = kpts1[j]
        color = 'lime' if is_correct else 'red'
        ax.plot([x0, x1 + w0], [y0, y1], color = color, lw = 0.5, alpha = 0.6)
        
    import matplotlib.patheffects as pe

    txt = ax.text(5, h - 10, f'TP={n_tp}  FP={n_fp}',
                  fontsize = 16, color = 'white')
    txt.set_path_effects([
        pe.withStroke(linewidth = 2.5, foreground = 'black')
    ])

---
## 1. Load per_keypoint.h5 — Build Per-Pair DataFrame

In [ ]:
def compute_pair_metrics(grp, reproj_thresholds = REPROJ_THRESHOLDS):
    """Compute metrics for ONE image-pair group in per_keypoint.h5.
    
    Returns a dict with metrics split by category: overall, static, dynamic.
    """
    gt0   = grp['gt_matches0'][:]
    pred0 = grp['pred_matches0'][:]
    obj0  = grp['object_ids0'][:]
    obj1  = grp['object_ids1'][:]
    re0   = grp['reproj_error0'][:]
    
    results = {}
    
    # Category masks
    categories = {
        'overall': np.ones(len(obj0), dtype = bool),
        'static':  np.isin(obj0, list(STATIC_IDS)),
        'dynamic': obj0 >= DYNAMIC_MIN,
    }
    
    categories2 = {
        'overall': np.ones(len(obj1), dtype = bool),
        'static':  np.isin(obj1, list(STATIC_IDS)),
        'dynamic': obj1 >= DYNAMIC_MIN,
    }
    
    for cat_name, cat_mask in categories.items():
        # Average number of keypoints in this category between view0 and view1
        n_kps = float(cat_mask.sum() + categories2[cat_name].sum()) / 2
        if n_kps == 0:
            results[f'{cat_name}_n_keypoints'] = 0
            results[f'{cat_name}_n_pred_matches'] = 0
            results[f'{cat_name}_n_gt_matchable'] = 0
            results[f'{cat_name}_n_tp'] = 0
            results[f'{cat_name}_n_fp'] = 0
            results[f'{cat_name}_precision'] = np.nan
            results[f'{cat_name}_recall'] = np.nan
            results[f'{cat_name}_mean_reproj'] = np.nan
            results[f'{cat_name}_median_reproj'] = np.nan
            results[f'{cat_name}_std_reproj'] = np.nan
            for t in reproj_thresholds:
                results[f'{cat_name}_acc@{t}px'] = np.nan
            continue
        
        g = gt0[cat_mask]
        p = pred0[cat_mask]
        r = re0[cat_mask]
        o0 = obj0[cat_mask]
        
        pred_matched = p >= 0
        gt_matchable = g >= 0
        correct = (p == g) & gt_matchable & pred_matched
        
        n_pred = int(pred_matched.sum())
        n_gt   = int(gt_matchable.sum())
        tp     = int(correct.sum())
        fp     = n_pred - tp
        
        precision = tp / n_pred if n_pred > 0 else np.nan
        recall    = tp / n_gt   if n_gt > 0   else np.nan
        
        # Object ID of the predicted match partner in view1
        partner_obj = np.full_like(o0, -1)
        partner_obj[pred_matched] = obj1[p[pred_matched]]
        same_object = pred_matched & (partner_obj == o0)
        
        # Reprojection error (only for predicted matches with finite error)
        re_pred = r[pred_matched]
        re_finite = re_pred[np.isfinite(re_pred)]
        mean_re   = float(np.mean(re_finite)) if len(re_finite) > 0 else np.nan
        median_re = float(np.median(re_finite)) if len(re_finite) > 0 else np.nan
        std_re    = float(np.std(re_finite)) if len(re_finite) > 0 else np.nan
        results[f'{cat_name}_n_keypoints']    = n_kps
        results[f'{cat_name}_n_pred_matches'] = n_pred
        results[f'{cat_name}_n_gt_matchable'] = n_gt
        
        results[f'{cat_name}_n_tp']           = tp
        results[f'{cat_name}_n_fp']           = fp
        
        # Missed matches analysis (only for predicted matches that are not correct but have a GT partner)
        wrong_matched = pred_matched & (~correct) & gt_matchable
        fp1 = wrong_matched & (r <= 3) & same_object
        fp2 = wrong_matched & (r <= 3) & (~same_object)
        fp3 = wrong_matched & (r > 3) & (~same_object)
        fp4 = wrong_matched & (r > 3) & same_object
        results[f'{cat_name}_fp1']         = fp1.sum()
        results[f'{cat_name}_fp2']         = fp2.sum()
        results[f'{cat_name}_fp3']         = fp3.sum()
        results[f'{cat_name}_fp4']         = fp4.sum()
        
        results[f'{cat_name}_precision']      = precision
        results[f'{cat_name}_recall']         = recall
        results[f'{cat_name}_mean_reproj']    = mean_re
        results[f'{cat_name}_median_reproj']  = median_re
        results[f'{cat_name}_std_reproj']     = std_re
        
        for t in reproj_thresholds:
            if n_pred > 0 and len(re_finite) > 0:
                # Match accuracy at various thresholds
                results[f'{cat_name}_acc@{t}px'] = float((re_finite <= t).sum()) / n_pred
            else:
                results[f'{cat_name}_acc@{t}px'] = np.nan
                
    return results


def objects_distribution(grp):
    """Return dynamic vs static object distribution per pair group."""
    obj0 = grp['object_ids0'][:]
    obj1 = grp['object_ids1'][:]
    n_static0 = np.isin(obj0, list(STATIC_IDS)).sum()
    n_dynamic0 = len(obj0) - n_static0
    n_static1 = np.isin(obj1, list(STATIC_IDS)).sum()
    n_dynamic1 = len(obj1) - n_static1
    return {
        'n_static': (n_static0 + n_static1) / 2,  # Average over the two views
        'n_dynamic': (n_dynamic0 + n_dynamic1) / 2,  # Average over the two views
    }


def load_all_results(RESULTS_FOLDER, model_dirs, reproj_thresholds = REPROJ_THRESHOLDS):
    """Load per_keypoint.h5 for every model and compute per-pair metrics."""
    rows = []
    
    for model_name, model_folder in model_dirs.items():
        h5_path = RESULTS_FOLDER / model_folder / 'per_keypoint.h5'
        if not h5_path.exists():
            print(f'  [WARN] {h5_path} not found')
            continue
        
        with h5py.File(h5_path, 'r') as f:
            for scene_id in sorted(f.keys()):
                scene_grp = f[scene_id]
                for pair_key in scene_grp:
                    grp = scene_grp[pair_key]
                    row = compute_pair_metrics(grp, reproj_thresholds)
                    row['model']    = model_name
                    row['scene']    = scene_id
                    row['pair_key'] = pair_key
                    pair_num = pair_key.split('_')[0]
                    row['pair_num'] = int(pair_num)
                    row['view_pair'] = '_'.join(pair_key.split('_')[1:])
                    row.update(objects_distribution(grp))
                    rows.append(row)
        
        print(f'  Loaded {model_name}: {len([r for r in rows if r["model"] == model_name])} entries')
    
    return pd.DataFrame(rows)


print('Loading results \u2026')
df = load_all_results(RESULTS_FOLDER, MODEL_DIRS)

---
## 2. Overall Results

#### Visualization of recall and precision values on the proposed dataset.

In [ ]:
fig, ax = plt.subplots(figsize = (6, 5))

rec  = df.groupby('model')['overall_recall'].mean()
prec = df.groupby('model')['overall_precision'].mean()

for model in MODEL_ORDER:
    if model in rec.index:
        ax.scatter(rec.loc[model], prec.loc[model], s = 140,
                    color = PALETTE[model], label = model,
                    zorder = 5, edgecolors = 'k', linewidth = 0.5)
        ax.annotate(model, (rec.loc[model], prec.loc[model]),
                    textcoords = 'offset points', xytext = (5, 5), fontsize = 10)

# F1 iso-lines
r_range = np.linspace(0.01, 1.0, 200)
for f1 in [0.3, 0.5, 0.7, 0.9]:
    p_range = f1 * r_range / (2 * r_range - f1)
    mask = (p_range >= 0) & (p_range <= 1)
    ax.plot(r_range[mask], p_range[mask], 'k--', lw = 0.5, alpha = 0.25)
    ax.annotate(f'F1={f1}',
                xy = (r_range[mask][-1], p_range[mask][-1]),
                fontsize = 8, color = 'gray')

ax.set_xlim(0, 1.05)
ax.set_ylim(0, 1.05)
ax.set_xlabel('GT Match Recall')
ax.set_ylabel('GT Match Precision')
ax.legend(fontsize = 10, frameon = False)

plt.tight_layout()
plt.show()

#### Detected keypoints, predicted matches comparison and detected keypoints with GT correspondences on the proposed dataset.

In [ ]:
df.groupby('model')[['overall_n_keypoints', 'overall_n_pred_matches', 'overall_n_gt_matchable']].mean()

#### Symmetric reprojection error breakdown on the proposed dataset.

In [ ]:
df.groupby('model')[['overall_acc@1px', 'overall_acc@3px', 'overall_acc@5px', 'overall_mean_reproj']].mean()

---
## 3. Static vs Dynamic

In [ ]:
# Reshape the original DataFrame to handle static vs dynamic object comparisons.

cats = ['overall', 'static', 'dynamic']
base_metrics = ['precision', 'recall', 'mean_reproj', 'median_reproj',
                'n_pred_matches', 'n_gt_matchable', 'n_tp', 'n_fp',
                'n_fp_reproj', 'n_fp_cross', 'fp1', 'fp2',
                'fp3', 'fp4', 'wrong5', 'n_keypoints', 'std_reproj',
                ] + [f'acc@{t}px' for t in REPROJ_THRESHOLDS]

long_rows = []
for _, row in df.iterrows():
    for cat in cats:
        r = {'model': row['model'], 'scene': row['scene'],
             'pair_key': row['pair_key'], 'category': cat}
        for m in base_metrics:
            r[m] = row.get(f'{cat}_{m}', np.nan)
        long_rows.append(r)

df_long = pd.DataFrame(long_rows)
# Drop rows where category has no keypoints at all
df_long = df_long[df_long['n_pred_matches'].notna() & (df_long['n_pred_matches'] > 0)].copy()

# Static/dynamic subset for plotting
cat_order = ['static', 'dynamic']
df_sd = df_long[df_long['category'].isin(cat_order)].copy()

#### Recall and precision values on static and dynamic objects.

In [ ]:
# Grouped bar chart: mean precision & recall — Static vs Dynamic
fig, axes = plt.subplots(1, 2, figsize = (14, 5))

for ax, metric, title in zip(
    axes, ['precision', 'recall'], ['Precision', 'Recall']
):
    agg = (
        df_sd.groupby(['model', 'category'])[metric]
        .mean()
        .reset_index()
    )

    sns.barplot(
        data = agg,
        x = 'model',
        y = metric,
        hue = 'category',
        order = MODEL_ORDER,
        hue_order = cat_order,
        ax = ax,
        palette = {'static': '#55A868', 'dynamic': '#C44E52'},
        edgecolor = 'k', lw = 0.5
    )

    for container in ax.containers:
        ax.bar_label(container, fmt = '%.2f', padding = 3, fontsize = 11, fontweight = 'bold')
        
    if ax is not axes[0]:
        ax.get_legend().remove()
    else:
        legend = ax.legend(title_fontsize = 11, frameon = False)
        for text in legend.get_texts():
            text.set_text(text.get_text().capitalize())

    ax.set_title(f'Mean {title}', fontsize = 24, fontweight = 'bold')
    ax.set_ylabel(title, fontsize = 17)
    ax.set_xlabel('')
    ax.tick_params(axis = 'x', labelsize = 14)
    ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

#### Symmetric reprojection error breakdown on static and dynamic objects.

In [ ]:
df_sd.groupby(['model', 'category'])[['acc@1px', 'acc@3px', 'acc@5px', 'mean_reproj', 'std_reproj']].mean().round(4)

#### Relationship between objects' pixels, keypoints, and predicted matches.

In [ ]:
def load_exr(path: str | Path, dtype = np.float32) -> np.ndarray:
    """Load a single-channel EXR file"""
    
    EXR = cv2.imread(str(path), cv2.IMREAD_ANYDEPTH | cv2.IMREAD_ANYCOLOR)
    if EXR is None:
        raise FileNotFoundError(f'[WARN] cv2.imread failed (file missing or unreadable): {path}')

    # Convert to float32 array
    EXR = np.asarray(EXR, dtype = dtype)

    # Handle EXR with channels (H, W, 1): take the first channel
    if EXR.ndim == 3 and EXR.shape[-1] >= 1:
        EXR = EXR[..., 0]

    return np.ascontiguousarray(EXR)

static = []
dynamic = []

scenes = df['scene'].unique()
for scene in scenes:
    
    scene_df = df[df['scene'] == scene]
    pair_nums = scene_df['pair_num'].unique()
    np.random.shuffle(pair_nums)
    
    for i, pair_num in enumerate(pair_nums):
        
        pair_df = scene_df[scene_df['pair_num'] == pair_num]
        view_pairs = pair_df['view_pair'].unique()
            
        for view_pair in view_pairs:
                
            view0, view1 = view_pair.split('_')
            mask0 = load_exr(DATASET_DIR / str(scene) / f'{pair_num:04d}' / f'obj_mask_for_view-{int(view0):04d}.exr')
            mask1 = load_exr(DATASET_DIR / str(scene) / f'{pair_num:04d}' / f'obj_mask_for_view-{int(view1):04d}.exr')
                
            avg_static = (np.isin(mask0, list(STATIC_IDS)).sum() + np.isin(mask1, list(STATIC_IDS)).sum()) / 2
            static.append(avg_static)
                
            avg_dynamic = (np.isin(mask0, list(STATIC_IDS), invert=True).sum() + np.isin(mask1, list(STATIC_IDS), invert=True).sum()) / 2
            dynamic.append(avg_dynamic)
    
            
print(f'Average static-object pixels per view: {np.mean(static):.1f}')
print(f'Average dynamic-object pixels per view: {np.mean(dynamic):.1f}')        
print(f'Ratio dynamic/static object pixels: {np.mean(dynamic) / np.mean(static):.4f}')

In [ ]:
df_kps = df.copy()
df_kps['dynamic_kps_ratio'] = df_kps['dynamic_n_keypoints'] / df_kps['overall_n_keypoints']
df_kps['static_kps_ratio'] = df_kps['static_n_keypoints'] / df_kps['overall_n_keypoints']
df_kps['dynamic_n_pred_matches_ratio'] = df_kps['dynamic_n_pred_matches'] / df_kps['overall_n_pred_matches']
df_kps['static_n_pred_matches_ratio'] = df_kps['static_n_pred_matches'] / df_kps['overall_n_pred_matches']
df_kps.groupby('model')[['static_kps_ratio', 'dynamic_kps_ratio', 'static_n_pred_matches_ratio', 'dynamic_n_pred_matches_ratio', 'static_n_pred_matches', 'dynamic_n_pred_matches']].mean().round(4)

#### Static vs Dynamic matches visualization.

In [ ]:
df_viz = (
    df.groupby(['scene', 'pair_key'])[['static_n_tp', 'dynamic_n_tp', 'dynamic_n_fp']]
    .mean()
    .reset_index()
    .rename(columns = {'static_n_tp': 'mean_static_n_tp_across_models', 'dynamic_n_tp': 'mean_dynamic_n_tp_across_models', 'dynamic_n_fp': 'mean_dynamic_n_fp_across_models'})
).dropna().reset_index(drop = True)

for _, prow in df_viz.sample(3).iterrows():
    scene, pk = prow['scene'], prow['pair_key']
    folder = pk.split('_')[0]
    v0, v1 = int(pk.split('_')[1]), int(pk.split('_')[2])

    img0 = np.array(Image.open(DATASET_DIR / scene / folder / f'render{v0}.png').convert('RGB'))
    img1 = np.array(Image.open(DATASET_DIR / scene / folder / f'render{v1}.png').convert('RGB'))

    fig, axes = plt.subplots(4, 2, figsize = (20, 24))

    for idx, model in enumerate(MODEL_ORDER):
        model_folder = MODEL_DIRS[model]
        with h5py.File(RESULTS_FOLDER / model_folder / 'per_keypoint.h5', 'r') as f:
            grp = f[scene][pk]
            kpts0 = grp['keypoints0'][:]
            kpts1 = grp['keypoints1'][:]
            pred_m0 = grp['pred_matches0'][:]
            gt_m0 = grp['gt_matches0'][:]
            obj_ids0 = grp['object_ids0'][:]

        for cat_idx, cat in enumerate(['static', 'dynamic']):
            
            ax = axes[idx, cat_idx]
            draw_separate_matches_on_pair(
                img0, img1, kpts0, kpts1, pred_m0, gt_m0, obj_ids0,
                ax = ax, max_lines = 300, title = model, cat = cat
            )
            
    plt.tight_layout()
    plt.show()

#### Matchable false positives breakdown.

In [ ]:
df_sd_fp = df_sd.copy()
df_sd_fp.dropna(subset = ['n_fp'], inplace = True)
df_sd_fp['missed_matches'] = (df_sd_fp['fp1']+ df_sd_fp['fp2'] + df_sd_fp['fp3'] + df_sd_fp['fp4'])
df_sd_fp['fp1_rate'] = df_sd_fp['fp1'] / df_sd_fp['missed_matches']
df_sd_fp['fp2_rate'] = df_sd_fp['fp2'] / df_sd_fp['missed_matches']
df_sd_fp['fp3_rate'] = df_sd_fp['fp3'] / df_sd_fp['missed_matches']
df_sd_fp['fp4_rate'] = df_sd_fp['fp4'] / df_sd_fp['missed_matches']

df_sd_fp.groupby(['model', 'category'])[['fp1_rate', 'fp2_rate', 'fp3_rate', 'fp4_rate']].mean().round(4)

---
## 4. Best & Worst Performing Pairs

Identify the pairs where models excel or struggle most, based on the F1 score.

In [ ]:
df_f1 = df.copy()
df_f1['dynamic_f1'] = 2 * (df['dynamic_precision'] * df['dynamic_recall']) / (df['dynamic_precision'] + df['dynamic_recall'] + 1e-8)
dyn_pair_mean = (
    df_f1.groupby(['scene', 'pair_key'])[['dynamic_f1', 'dynamic_precision', 'dynamic_recall']]
    .mean()
    .reset_index()
    .rename(columns = {'dynamic_f1': 'mean_dyn_f1_across_models', 'dynamic_precision': 'mean_dyn_precision_across_models', 'dynamic_recall': 'mean_dyn_recall_across_models'})
    .sort_values('mean_dyn_f1_across_models', ascending = False)
)

print('Pairs EASIEST for DYNAMIC objects across all models (highest mean dynamic F1 score):')
display(dyn_pair_mean.head(10))
print('\nPairs HARDEST for DYNAMIC objects across all models (lowest mean dynamic F1 score):')
display(dyn_pair_mean.tail(10))

In [ ]:
N_DYN_VIS = 3

dyn_easiest = dyn_pair_mean.dropna().head(N_DYN_VIS)

for _, prow in dyn_easiest.iterrows():
    scene, pk = prow['scene'], prow['pair_key']
    folder = pk.split('_')[0]
    v0, v1 = int(pk.split('_')[1]), int(pk.split('_')[2])

    img0 = np.array(Image.open(DATASET_DIR / scene / folder / f'render{v0}.png').convert('RGB'))
    img1 = np.array(Image.open(DATASET_DIR / scene / folder / f'render{v1}.png').convert('RGB'))

    fig, axes = plt.subplots(2, 2, figsize = (20, 12))

    for idx, model in enumerate(MODEL_ORDER):
        model_folder = MODEL_DIRS[model]
        with h5py.File(RESULTS_FOLDER / model_folder / 'per_keypoint.h5', 'r') as f:
            grp = f[scene][pk]
            kpts0 = grp['keypoints0'][:]
            kpts1 = grp['keypoints1'][:]
            pred_m0 = grp['pred_matches0'][:]
            gt_m0 = grp['gt_matches0'][:]
            obj_ids0 = grp['object_ids0'][:]

        ax = axes[idx//2, idx%2]
        draw_matches_on_pair(
            img0, img1, kpts0, kpts1, pred_m0, gt_m0, obj_ids0,
            category = 'dynamic', ax = ax, max_lines = 300,
            title = f'{model} \u2014 Dynamic Matches'
        )

    plt.tight_layout()
    plt.show()